In [1]:
import json                       # saves the metrics as a JSON file
from pathlib import Path          # handle file and folder paths

import nltk                       # Natural Language Toolkit
import numpy as np                # numerical arrays
import pandas as pd               # tables (Data frame)
import plotly.express as px       # interactive charts 
from nltk.sentiment import SentimentIntensityAnalyzer     # VADER sentiment analyser
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix   # evaluation tool

pd.set_option("display.max_colwidth", 120)    # show long ticket text without cutting it off

In [2]:
DATA_DIR = Path("../data") if Path("../data/train.csv").exists() else Path("data")   # find the data folder (one level up, or here)
MODEL_DIR = DATA_DIR.parent / "models"                # saved results go in the "models folder next to "data"
MODEL_DIR.mkdir(exist_ok=True)                   #create it if it does not exist
print("Using data folder:", DATA_DIR.resolve())   # show which folder was found


train_df = pd.read_csv(DATA_DIR / "train.csv")     # training tickets (used only for tuning)
test_df = pd.read_csv(DATA_DIR / "test.csv")       # test tickets (used only for the final score)
all_df = pd.read_csv(DATA_DIR / "tickets_clean.csv")  # every cleaned ticket


SENTIMENTS = ["Negative", "Neutral", "Positive"]   # the three sentiment labels, in a fixed order
print("Train:", len(train_df), "| Test:", len(test_df), "| ALL:", len(all_df))   # dataset size
test_df[["text", "sentiment"]].head()
      


Using data folder: C:\Users\SREEHARIB\data
Train: 770 | Test: 193 | ALL: 963


,text,sentiment
0,"Good morning,I have a question. I would like a refund for order 31832.",Neutral
1,Please check this for me. I waited an hour on the phone nobody helped me.,Neutral
2,"I love your product, but The software kepps disconnecting from the server, Pespond ASAP.",Positive
3,"Great app overall, however I was charged twice for my subscription this month.There is an incorrect amount on my lat...",Positive
4,"Great app overall, however i forgot my password and the reset email never arrives. This is an emergency.",Positive


In [3]:
## load the VADER analyzer

In [3]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download("vader_lexicon", quiet=True)   # download the sentiment dictionary

sia = SentimentIntensityAnalyzer()           # create the VADER analyser

print("VADER ready. Dictionary size:", len(sia.lexicon), "words")  # confirm that it loaded


VADER ready. Dictionary size: 7502 words


In [5]:
## See how VADER scores a message

In [4]:
examples = [                                   # FIVE EXAMPLES
    "Thank you so much, the support team was great!",
    "Hello, I have a question about my invoice."
    "This is unacceptable, I am extremely upset!!!",
    "The app is not good.",
    "THE APP IS NOT GOOD!!!",
]

for text in examples:                         # go through them one by one
    scores = sia.polarity_scores(text)        # neg, neu, pos, and compound score   
    print(f"{text}\n    {scores}\n")           # print the text and its scores

Thank you so much, the support team was great!
    {'neg': 0.0, 'neu': 0.378, 'pos': 0.622, 'compound': 0.8707}

Hello, I have a question about my invoice.This is unacceptable, I am extremely upset!!!
    {'neg': 0.429, 'neu': 0.571, 'pos': 0.0, 'compound': -0.7763}

The app is not good.
    {'neg': 0.376, 'neu': 0.624, 'pos': 0.0, 'compound': -0.3412}

THE APP IS NOT GOOD!!!
    {'neg': 0.451, 'neu': 0.549, 'pos': 0.0, 'compound': -0.5076}



In [7]:
# Function that turns a score into label

In [5]:
def analyze_sentiment(text, pos_thr=0.05, neg_thr=-0.05):
    # returns (label, compound score) for one ticket

    compound = sia.polarity_scores(str(text))["compound"]

    if compound >= pos_thr:
        return "Positive", compound

    elif compound <= neg_thr:
        return "Negative", compound

    else:
        return "Neutral", compound


# Quick tests
print(analyze_sentiment("Thanks for the great service so far."))
print(analyze_sentiment("I have a question."))
print(analyze_sentiment("This is the worst experience."))

('Positive', 0.7906)
('Neutral', 0.0)
('Negative', -0.6249)


In [9]:
# Predict sentiment for the test set

In [6]:
scored = test_df["text"].apply(analyze_sentiment)           # (label, score) pair for every ticket

results = test_df[["ticket_id", "text", "sentiment"]].copy()   # start from the true labels
results = results.rename(columns={"sentiment": "actual"})   # call the true label "actual"
results["predicted"] = [s[0] for s in scored]               # VADER's label
results["compound"] = [round(s[1], 3) for s in scored]      # VADER's score
results["correct"] = results["actual"] == results["predicted"]   # True when VADER agrees with the label
results.head(8)                                             # show the first 8 rows

,ticket_id,text,actual,predicted,compound,correct
0,T01764,"Good morning,I have a question. I would like a refund for order 31832.",Neutral,Positive,0.660,False
1,T01289,Please check this for me. I waited an hour on the phone nobody helped me.,Neutral,Positive,0.318,False
2,T00591,"I love your product, but The software kepps disconnecting from the server, Pespond ASAP.",Positive,Positive,0.382,True
3,T01834,"Great app overall, however I was charged twice for my subscription this month.There is an incorrect amount on my lat...",Positive,Positive,0.634,True
4,T00328,"Great app overall, however i forgot my password and the reset email never arrives. This is an emergency.",Positive,Positive,0.361,True
5,T01827,This is unacceptable. I cannot find my invoice for order 87942.,Negative,Negative,-0.459,True
6,T00585,I am really frustrated. Can I get a refund for the duplicate purchase?,Negative,Negative,-0.571,True
7,T01946,This is the wrost experience. I cannot find my invoice for order 98944.,Negative,Neutral,0.000,False


In [11]:
## Accuracy, precision, recall and F1 score

In [7]:
y_true = results["actual"]    # the true sentiments
y_pred = results["predicted"]    # VADER's prediction

acc = accuracy_score(y_true, y_pred)    # share of correct predictions
print(f"VADER accuracy on the test set: {acc: .1%}\n")   # headline number

print(classification_report(y_true, y_pred, labels=SENTIMENTS, zero_division=0))   # precision, recall, F1 per sentiment

VADER accuracy on the test set:  68.4%

              precision    recall  f1-score   support

    Negative       0.88      0.84      0.86        93
     Neutral       0.70      0.27      0.39        59
    Positive       0.47      0.93      0.62        41

    accuracy                           0.68       193
   macro avg       0.68      0.68      0.62       193
weighted avg       0.73      0.68      0.66       193



In [13]:
# Confusion matrix

In [8]:
cm = confusion_matrix(y_true, y_pred, labels=SENTIMENTS)    # count of (actual, predicted) pairs

fig = px.imshow(cm, x=SENTIMENTS, y= SENTIMENTS, text_auto=True,    # heatmap with number inside
                aspect="auto",                            # stretch the cell to fill the chart
                color_continuous_scale="Purples",   # darker = more tickets
                labels=dict(x="Predicted sentiment", y="Actual sentiment", color="Tickets"),  #axis names
                title="Sentiment confusion market (test set)")   # chart title
fig.update_layout(width=700, height=500)         # chart size
fig.show()           # draw the chart


In [15]:
# Error analysis

In [9]:
print("Average compound score by TRUE sentiment:") # heading
print(results.groupby("actual")["compound"].mean().round(3).reindex(SENTIMENTS))     # mean score per true label

errors = results[~results["correct"]]   # keep only the wrong prediction
print(f"\nVADER got {len(errors)} of {len(results)} tickets wrong. Examples:")   #how many mistakes
errors[["text", "actual", "predicted", "compound"]].sample(min(8, len(errors)), random_state=1)  # 8 random mistake

Average compound score by TRUE sentiment:
actual
Negative   -0.435
Neutral     0.159
Positive    0.543
Name: compound, dtype: float64

VADER got 61 of 193 tickets wrong. Examples:


,text,actual,predicted,compound
127,"Good morning,I have a question. I cannot find my invoice for order 41630.",Neutral,Positive,0.440
129,Please check this for me. My account was hacked and I cannot access it. I need this resolved immediately.,Neutral,Positive,0.122
7,This is the wrost experience. I cannot find my invoice for order 98944.,Negative,Neutral,0.000
158,This is the wrost experience. My credit card payment failed but money was deducted. I need this resolved immediately.,Negative,Positive,0.241
169,"Good morning,I have a question. An unknown charge appeared on my statement.",Neutral,Positive,0.440
74,"Hello,Hi team, The verification code for two factor login is not working. Please fix this urgently.",Neutral,Negative,-0.241
103,Please check this for me. The product quality is much wrose than advertised. I need this resolved immediately.,Neutral,Positive,0.490
108,This is unacceptable. My credit card payment failed but money was deducted. Please fix this urgently.,Negative,Positive,0.153


In [17]:
## Tune the thresholds on the TRAINING data

In [10]:
train_scores = train_df["text"].apply(lambda t: sia.polarity_scores(str(t))["compound"])   # compound score for every training ticket

best = (0.05, -0.05, 0.0)                                   # best (pos_thr, neg_thr, accuracy) so far
for pos_thr in np.arange(0.05, 0.61, 0.05):                 # try positive cut-offs from 0.05 to 0.60
    for neg_thr in np.arange(-0.05, -0.61, -0.05):          # try negative cut-offs from -0.05 to -0.60
        pred = np.where(train_scores >= pos_thr, "Positive",   # label every training ticket with this pair
                        np.where(train_scores <= neg_thr, "Negative", "Neutral"))
        a = (pred == train_df["sentiment"].values).mean()   # accuracy of this pair on the training data
        if a > best[2]:                                     # keep it if it is the best so far
            best = (round(float(pos_thr), 2), round(float(neg_thr), 2), float(a))

tuned_pos, tuned_neg, train_acc = best                      # unpack the winner
tuned_pred = results["compound"].apply(                     # apply the tuned cut-offs to the TEST scores
    lambda c: "Positive" if c >= tuned_pos else ("Negative" if c <= tuned_neg else "Neutral"))
tuned_acc = accuracy_score(y_true, tuned_pred)              # test accuracy with the tuned cut-offs

print(f"Best cut-offs on training data: positive >= {tuned_pos}, negative <= {tuned_neg}  (train accuracy {train_acc:.1%})")
print(f"Test accuracy with default cut-offs (+-0.05): {acc:.1%}")   # before tuning
print(f"Test accuracy with tuned cut-offs           : {tuned_acc:.1%}")   # after tuning

Best cut-offs on training data: positive >= 0.6, negative <= -0.1  (train accuracy 75.2%)
Test accuracy with default cut-offs (+-0.05): 68.4%
Test accuracy with tuned cut-offs           : 79.3%


In [19]:
## Sentiment distribution

In [11]:
all_scored = all_df["text"].apply(analyze_sentiment)        # (label, score) for every ticket
all_df["pred_sentiment"] = [s[0] for s in all_scored]       # VADER's label
all_df["compound"] = [round(s[1], 3) for s in all_scored]   # VADER's score

dist = pd.DataFrame({                                       # side-by-side counts
    "True label": all_df["sentiment"].value_counts(),       # how many of each true sentiment
    "VADER prediction": all_df["pred_sentiment"].value_counts(),   # how many of each predicted sentiment
}).reindex(SENTIMENTS).fillna(0).astype(int)                # fixed order, no missing values
display(dist)                                               # show the table

dist_long = dist.reset_index().melt(id_vars="index", var_name="source", value_name="tickets")   # long format for plotting
px.bar(dist_long, x="index", y="tickets", color="source", barmode="group",   # grouped bar chart
       labels={"index": "Sentiment"}, title="Sentiment distribution: true labels vs VADER").show()   # draw it

,True label,VADER prediction
Negative,470,488
Neutral,265,81
Positive,228,394


In [21]:
## Sentiment by category

In [12]:
by_cat = (all_df.groupby(["category", "pred_sentiment"]).size()     # count tickets per category and sentiment
          .reset_index(name="tickets"))           # trun the counts into table
px.bar(by_cat, x="category", y="tickets", color="pred_sentiment", barmode="stack",    # stacked bar chart
       category_orders={"pred_sentiment": SENTIMENTS},    # fixed legend order
       color_discrete_map={"Negative": "#e5484d", "Neutral": "#8b8d98", "Positive": "#30a46c"},    # red, grey, green
       title="Predicted sentiment by ticket category").update_xaxes(tickangle=20).show()   # tilt long labels, draw it

In [23]:
# Sentiment trend over time

In [13]:
rng = np.random.default_rng(42)                             # fixed seed so the dates are repeatable
today = pd.Timestamp.today().normalize()                    # today's date without the time
all_df["created_at_simulated"] = today - pd.to_timedelta(rng.integers(0, 30, len(all_df)), unit="D")   # random day in the last 30 days

daily = (all_df.groupby(["created_at_simulated", "pred_sentiment"]).size()   # tickets per day and sentiment
         .reset_index(name="tickets"))                      # turn the counts into a table

px.line(daily, x="created_at_simulated", y="tickets", color="pred_sentiment", markers=True,   # one line per sentiment
        category_orders={"pred_sentiment": SENTIMENTS},     # fixed legend order
        color_discrete_map={"Negative": "#e5484d", "Neutral": "#8b8d98", "Positive": "#30a46c"},   # red, grey, green
        labels={"created_at_simulated": "Date (simulated)"},   # axis name makes the simulation obvious
        title="Sentiment trend over time (SIMULATED dates)").show()   # draw it


In [25]:
# Save the result

In [14]:
all_df.to_csv(DATA_DIR / "sentiment_results.csv", index=False)   # every ticket with its sentiment

report = classification_report(y_true, y_pred, labels=SENTIMENTS, output_dict=True, zero_division=0)   # scores as a dictionary
metrics = {                                                 # numbers to keep for the report
    "accuracy_default_thresholds": float(acc),              # accuracy with +-0.05
    "accuracy_tuned_thresholds": float(tuned_acc),          # accuracy with the tuned cut-offs
    "tuned_positive_threshold": tuned_pos,                  # tuned positive cut-off
    "tuned_negative_threshold": tuned_neg,                  # tuned negative cut-off
    "labels": SENTIMENTS,                                   # label order
    "confusion_matrix": cm.tolist(),                        # confusion matrix as a list
    "report": report,                                       # precision, recall, F1 per sentiment
    "n_test": int(len(test_df)),                            # test set size
}
(MODEL_DIR / "sentiment_metrics.json").write_text(json.dumps(metrics, indent=2))   # save the metrics
print("Saved:", DATA_DIR / "sentiment_results.csv", "and", MODEL_DIR / "sentiment_metrics.json")   # confirmation

Saved: ..\data\sentiment_results.csv and ..\models\sentiment_metrics.json
